### Exploratory Data Analysis (EDA)

This notebook serves to conduct exploratory data analysis on the dataset.

### 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pandas as pd
from wordcloud import WordCloud, STOPWORDS
import warnings
import re
from langdetect import detect, LangDetectException


warnings.filterwarnings('ignore')

### 2. Loading Dataset

In [ ]:
data_path = Path('data/raw/fake_job_postings.csv')
df_source = pd.read_csv(data_path)
df = df_source.copy(deep=True)

print(df.shape)
print(df.info())
print(df.describe())
print(df.columns.tolist())


>The dataset has `17,880 rows` and `18 columns`.
>
>The columns are `job_id`, `title`, `location`, `department`, `salary_range`, `company_profile`, `description`, `requirements`, `benefits`, `telecommuting`, `has_company_logo`, `has_questions`, `employment_type`, `required_experience`, `required_education`, `industry`, `function`, `fraudulent`.
>
>There are `5 numerical int64 columns` (job_id, telecommuting, has_company_logo, has_questions, fraudulent), while the rest are text str columns.

In [ ]:
df.head()

### 3. Data Cleaning

This section serves to clean the dataset by handling inconsistencies in strings, null or empty values, etc.


In [ ]:
df.isnull().sum().sort_values(ascending=False)


In [ ]:
df.isnull().mean() * 100

Given that salary range has 84.0% missing values and department has 64.5% missing values, can consider dropping these features since they are too sparse to extract meaning. We will transform `salary_range` to `salary_provided boolean column` instead.

In [ ]:
placeholders = ['n/a', 'na', 'none', 'null', '-', '?', 'not specified', 'unknown', 'other']
for col in df.select_dtypes(include='object').columns:
    mask = df[col].str.strip().str.lower().isin(placeholders)
    count = mask.sum()
    if count > 0:
        print(f"column name({col}): {count} placeholder values")
        print(df[col][mask].value_counts())
        print('\n')

In [ ]:
df['benefits'] = df['benefits'].replace('na', np.nan)
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
str_cols =  ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']
num_cols = ['job_id', 'telecommuting', 'has_company_logo', 'has_questions', 'fraudulent']
text_cols = ['company_profile', 'description', 'requirements', 'benefits']
cat_cols = ['location', 'employment_type', 'required_experience', 
            'required_education', 'industry', 'function']
binary_cols = ['telecommuting', 'has_company_logo', 'has_questions']
target = 'fraudulent'
to_drop = ['job_id', 'department', 'salary_range']

df[text_cols] = df[text_cols].fillna('')
df[cat_cols] = df[cat_cols].fillna('Not Provided')

df.isnull().sum()

We will `drop job_id column` as ids do not have any predictice power over whether selected job is fradulent

In [ ]:
df = df.drop(['job_id'], axis=1)
df.info()

given that the location column can be split into `country`, `state`, `city`, We will create 3 additional columns country, state and city.

In [ ]:
df['location'].head()
df[["country","state","city"]] = df["location"].str.split(",", n=2, expand=True)

#top 5 countries
df["country"].value_counts().head(5)

we count the number of values with missing states or cities.

In [ ]:
df[["state", "city"]].isna().sum()

In [ ]:
df.head()

### 4. Univariate Analysis

#### (a) Numerical Column Analysis 

Analyze the distributions and spread of numerical data in the numerical data columns.

In [ ]:
df.describe().T

>Only `telecommuting` , `has_company_logo`, `has_questions` and `fradulent `columns are numeric and since we have dropped the job_id column, these four are the only columns we can do statistical description of its data spread.

##### Telecommunting

In [ ]:
print("\nTelecommuting distribution:")
print(df['telecommuting'].value_counts(normalize=True) * 100)

sns.countplot(x=df['telecommuting'])
plt.title('Distribution of Job Postings with Telecommuting Options')
plt.show()

>we can see that most jobs `do not involve telecommunting`

##### Company Logo

In [ ]:
print("\nHas Company Logo distribution:")
print(df['has_company_logo'].value_counts(normalize=True) * 100)

sns.countplot(x=df['has_company_logo'])
plt.title('Distribution of Job Postings with Company Logo')
plt.show()

> Majority of job posting `do have a company logo`

##### Telecommunting

In [ ]:
print("\nHas Questions distribution:")
print(df['has_questions'].value_counts(normalize=True) * 100)

sns.countplot(x=df['has_questions'])
plt.title('Distribution of Job Postings with Questions')
plt.show()

>`Roughly even split` in whether screening questions are present in the job posting.

In [ ]:
print("\nFraudulent distribution:")
print(df['fraudulent'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6,4))
sns.countplot(x='fraudulent',data=df)
plt.title('Distribution of Fraudulent vs Non-Fraudulent Job Postings')
plt.show()

>majority of all jobs are not fradulent

#### (b) Categorical Column Analysis

Analyze how different categories in each column lead to proportions of job postings to be fradulent or not.

Serves to capture relationships on where certain types of categories have a larger percentage of fraduluent job postings.

In [ ]:
print(f"Number of employment types: {df['employment_type'].value_counts().size}")

##### Employment Type

In [ ]:
## Comapre employment type distribution for fraudulent job postings

plt.figure(figsize=(10,6))
order = df.groupby('employment_type')['fraudulent'].sum().sort_values(ascending=False).index
sns.barplot(x='employment_type', y='fraudulent', data=df, palette='viridis', estimator=sum, order=order)
plt.title(f'Fraudulent Count by employment_type')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
order = df.groupby('employment_type')['fraudulent'].mean().sort_values(ascending=False).index
sns.barplot(x='employment_type', y='fraudulent', data=df, palette='viridis', order=order)
plt.title(f'Fraudulent Rate by employment_type')
plt.xticks(rotation=45)
plt.show()

>Majority of fraudulent job postings are for `full time positions`.
>
>However, we see that part-time job postings have the `greatest proportion of fraudulent job postings`.

##### Required Experience

In [ ]:
print(f"Number of required experience levels: {df['required_experience'].value_counts().size}")

In [ ]:
## Comapre required_experience distribution for fraudulent job postings

plt.figure(figsize=(10,6))
order = df.groupby('required_experience')['fraudulent'].sum().sort_values(ascending=False).index
sns.barplot(x='required_experience', y='fraudulent', data=df, palette='viridis', estimator=sum, order=order)
plt.title(f'Fraudulent Count by required_experience')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
order = df.groupby('required_experience')['fraudulent'].mean().sort_values(ascending=False).index
sns.barplot(x='required_experience', y='fraudulent', data=df, palette='viridis', order=order)
plt.title(f'Fraudulent Rate by required_experience')
plt.xticks(rotation=45)
plt.show()

> Majority of the job postings that `do not highlight a required experience` are fraudulent.
>
> However, majority of executive job postings are seen to have a `higher percentage` of fraudulent postings.

##### Required Education

In [ ]:
print(f"Number of required education levels: {df['required_education'].value_counts().size}")

In [ ]:
## Compare required_education distribution for fraudulent job postings
plt.figure(figsize=(10,10))
order = df.groupby('required_education')['fraudulent'].sum().sort_values(ascending=False).index
sns.barplot(y='required_education', x='fraudulent', data=df, palette='viridis', estimator=sum, order=order)
plt.title(f'Fraudulent Count by required_education')
plt.xticks(rotation=90)
plt.show()

In [ ]:
plt.figure(figsize=(10,10))
order = df.groupby('required_education')['fraudulent'].mean().sort_values(ascending=False).index
sns.barplot(y='required_education', x='fraudulent', data=df, palette='viridis', order=order)
plt.title(f'Fraudulent Count by required_education')
plt.xticks(rotation=90)
plt.show()

>Majority of fraudulent postings `do not have a specified education`.
>
>However, a significant proportion of job postings that `only required high school education` are considered fraudulent.

##### Function

In [ ]:
print(f"Number of functions: {df['function'].value_counts().size}")

38 function categories

In [ ]:
## function distribution for fraudulent job postings
plt.figure(figsize=(10,10))
order = df.groupby('function')['fraudulent'].sum().sort_values(ascending=False).index
sns.barplot(y='function', x='fraudulent', data=df, palette='viridis', estimator=sum, order=order)
plt.title(f'Fraudulent Count by function')
plt.xticks(rotation=90)
plt.show()

In [ ]:
plt.figure(figsize=(10,10))
order = df.groupby('function')['fraudulent'].mean().sort_values(ascending=False).index
sns.barplot(y='function', x='fraudulent', data=df, palette='viridis', order=order)
plt.title(f'Fraudulent Count by function')
plt.xticks(rotation=90)
plt.show()

> Majority of fraudulent postings `do not have a specified function`
> 
> However, `administraive functions` have the highest proportion of fraudulent job postings.

##### Industry

In [ ]:
print(f"Number of industries: {df['industry'].value_counts().size}")

In [ ]:
# Calculate fraud rate per industry
ind_fraud = df.groupby('industry')['fraudulent'].mean().sort_values(ascending=False)

# Filter for industries that actually have a significant number of postings (e.g., > 50)
popular_industries = df['industry'].value_counts()
high_volume_ind = popular_industries[popular_industries > 50].index

print(ind_fraud[high_volume_ind].head(10)) # Plot only the high-volume, high-risk industries

> There is not a clear relationship between which industry/null values creates the most fradulent job postings

##### Country

In [ ]:
print(f"No. of Countries: {df['country'].value_counts().size}")

In [ ]:
top_fraud_countries = df[df['fraudulent'] == 1]['country'].value_counts().head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top_fraud_countries.values, y=top_fraud_countries.index, palette='flare')
plt.title('Top 10 Countries with Highest TOTAL Fake Postings')
plt.xlabel('Number of Fake Postings')
plt.show()

> We can see that `US`, `Australia` and `Great Britain` have the highest proportions of fake postings

##### State, City

In [ ]:
print(f"No. of States: {df['state'].value_counts().size}")

print(f"No. of Cities: {df['city'].value_counts().size}")

In [ ]:
df['city'].value_counts().head(10)

In [ ]:
# Check if BOTH State and City are missing
df['vague_location'] = (df['state'].isna()) & (df['city'].isna())
df['vague_location'] = df['vague_location'].astype(int)

# Check the Fraud Rate
vague_rate = df.groupby('vague_location')['fraudulent'].mean()
print("Fraud Rate: Vague (1) vs Specific (0) Locations:")
print(vague_rate)

> We can see that job postings `missing both state and city` have a higher percentage of being fake

#### (c) Free Text Column Analaysis

Preprocess the texts by removing common punctuations via Regex and common stop words to prevent miscalculation on the importance of seen words.

Use word cloud to give a basic visualisation of the spread of the words in the free text.

Use TF-IDF to give statistics on the contextual importance of the words used in these free text.



In [ ]:
df['title'].head(10)

In [ ]:
print(STOPWORDS)
custom_stopwords = STOPWORDS.union({"will", "want", "able","amp", "#URL", "link", "url"})

def plot_wordcloud(text_data,title,color):
    
    combined_text = " ".join(text_data)
    wc =  WordCloud(
        width=800,
        height=400,
        colormap=color,
        stopwords=custom_stopwords
    ).generate(combined_text)
    
    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
fake_job_titles = df[df['fraudulent'] == 1]['title']
real_job_titles = df[df['fraudulent'] == 0]['title']

fake_job_profile = df[df['fraudulent'] == 1]['company_profile']
real_job_profile = df[df['fraudulent'] == 0]['company_profile']

fake_job_description = df[df['fraudulent'] == 1]['description']
real_job_description = df[df['fraudulent'] == 0]['description']

fake_job_requirements = df[df['fraudulent'] == 1]['requirements']
real_job_requirements = df[df['fraudulent'] == 0]['requirements']

fake_job_benefits = df[df['fraudulent'] == 1]['benefits']
real_job_benefits = df[df['fraudulent'] == 0]['benefits']

##### Job Titles

In [ ]:
plot_wordcloud(fake_job_titles, 'Word Cloud of Fake Job Titles',"Reds")
plot_wordcloud(real_job_titles, 'Word Cloud of Real Job Titles',"Greens")

>We can see that both fake and real job postings often `commonly feature the same titles` such as customer service, engineer so it may not be as useful as a feature in our model. However, `data entry jobs have a higher probability of being fake`, while `teachers have a higher probability of being real`

##### Company Profiles

In [ ]:
plot_wordcloud(fake_job_profile, 'Word Cloud of Fake Job Company Profiles',"Reds")

plot_wordcloud(real_job_profile, 'Word Cloud of Real Job Company Profiles',"Greens")

Job postings with the word `team`is highly likely to be real

##### Job Requirements

In [ ]:
plot_wordcloud(fake_job_requirements, 'Word Cloud of Fake Job Requirements',"Reds")

plot_wordcloud(real_job_requirements, 'Word Cloud of Real Job Requirements',"Greens")

##### Job Description

In [ ]:
plot_wordcloud(fake_job_description, 'Word Cloud of Fake Job Description',"Reds")

plot_wordcloud(real_job_description, 'Word Cloud of Real Job Description',"Greens")

##### TFIDF of all Free Text
we combine all the text columns to analyse the top words in both real and fake jobs across all freee texts

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import pandas as pd

# We combine the text columns into one 'all_text' column
df['all_text'] = df['company_profile'] + " " + df['description'] + " " + df['requirements'] + " " + df['benefits']
base_stop_words = set(ENGLISH_STOP_WORDS)
custom_stopwords = list(base_stop_words | set(custom_stopwords))

# Initialize Vectorizer
# We use max_features=5000 to keep the most important words and ignore rare typos
tfidf = TfidfVectorizer(stop_words=custom_stopwords, max_features=5000, ngram_range=(1, 2))

# Create the TF-IDF Matrix
tfidf_matrix = tfidf.fit_transform(df['all_text'])

# Convert to a DataFrame for analysis
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())


In [ ]:
# Separate the TF-IDF scores by label
fake_tfidf = tfidf_df[df['fraudulent'] == 1]
real_tfidf = tfidf_df[df['fraudulent'] == 0]

# Calculate the mean score for every word in each category
top_fake_words = fake_tfidf.mean().sort_values(ascending=False).head(20)
top_real_words = real_tfidf.mean().sort_values(ascending=False).head(20)

print("Top Keywords in Fake Jobs:")
print(top_fake_words)



In [ ]:
print("Top Keywords in Real Jobs:")
print(top_real_words)

>The most common words found in fraudulent job postings typically use `simplistic words` like work, data, skills with `little to no specificity to the job` that it is >posting about. Majority of the postings also indicate that it is an entry level job. As such, many of the fraudulent job postings have `simplistic descriptive words` >used and have very little role in explaining the specificity of the job.
>
>In contrast, real job postings are more `role based` and `placing emphasis on teams, business, development`. Key business terms such as marketing, sales and customers are more prevalent in real job postings as well.

##### Text Length Analysis

We analyse the text lenght of all text-based columns to find any useful correlation

In [ ]:
for col in text_cols:
    df[f'{col}_len'] = df[col].str.len()
    df[f'{col}_word_count'] = df[col].str.split().str.len().fillna(0).astype(int)

len_cols = [f'{col}_len' for col in text_cols]

for col in len_cols:
    print(f"\n{col}:")
    print(df.groupby('fraudulent')[col].describe().round(1))

**Company Profile Length:**
- Legitimate postings: mean = 640.8 chars, median = 588.0 chars.
- Fraudulent postings: mean = 230.9 chars, median = 0.0 chars.
- A median of 0 means over half of fraudulent postings have an empty company profile. This is the strongest text-based signal.

**Requirements Length:**
- Legitimate: mean = 597.5, median = 476.5.
- Fraudulent: mean = 446.0, median = 249.0.
- Fraudulent postings provide roughly half the detail in requirements (249.0 / 476.5 = 0.52).

**Description Length:**
- Legitimate: mean = 1221.2, median = 1027.0.
- Fraudulent: mean = 1154.8, median = 844.5.
- Relatively similar so this is a weak signal (844.5 / 1027.0 = 0.82).

**Benefits Length:**
- Legitimate: mean = 208.7, median = 47.0.
- Fraudulent: mean = 212.2, median = 36.0.
- Nearly identical so another weak signal.

**Conclusion:** `company_profile_len` and `requirements_len` are the strongest text-length features. This reinforces the pattern that fraudulent postings appear to omit information. A combined `missing_count` or `total_text_length` feature should capture this signal.

### Feature Engineering 

This section serves to engineer new features to discover relationships through the interactions between features.

In [ ]:
check_cols = ['company_profile', 'description', 'requirements', 'benefits',
              'employment_type', 'required_experience', 'required_education',
              'industry', 'function']

df['missing_count'] = 0
for col in check_cols:
    if col in text_cols:
        df['missing_count'] += (df[col].str.strip() == '').astype(int)
    elif col in cat_cols:
        df['missing_count'] += (df[col] == 'Not Provided').astype(int)

print(pd.crosstab(df['missing_count'], df['fraudulent'], margins=True)) #out of the 9 columns in check_cols, how many are missing
print()
fraud_by_missing = df.groupby('missing_count')['fraudulent'].mean() * 100 #fraud rate for each amount of missing values
print(fraud_by_missing)

In [ ]:
df['total_text_len'] = df['company_profile_len'] + df['description_len'] + df['requirements_len'] + df['benefits_len']

print(df.groupby('fraudulent')['total_text_len'].describe().round(1))

In [ ]:
df['salary_provided'] = df['salary_range'].notna().astype(int)

ct = pd.crosstab(df['salary_provided'], df['fraudulent'], margins=True)
ct['fraud_rate'] = ct[1] / ct['All'] * 100
ct['fraud_share'] = ct[1] / 866 * 100
print(ct)

In [ ]:
df['has_company_profile'] = (df['company_profile'].str.strip() != '').astype(int)

ct = pd.crosstab(df['has_company_profile'], df['fraudulent'], margins=True)
ct['fraud_rate'] = ct[1] / ct['All'] * 100
ct['fraud_share'] = ct[1] / 866 * 100
print(ct)

#### Feature Engineering Summary

**missing_count:**
- Counts the number of empty/unprovided fields across 9 columns (4 text + 5 categorical).
- Fraud rate generally increases with more missing fields: 3.61% at 0 missing, 9.35% at 4, 10.89% at 6, 18.40% at 8.
- Not perfectly linear, certain combinations of missing fields matter more than others but the overall trend is clear.

**total_text_len:**
- Sum of character lengths across company_profile, description, requirements, benefits.
- Fraudulent median: 1,624.5 chars vs legitimate median: 2,530.0 chars (1,624.5 / 2,530.0 = 0.64).
- Confirms fraudulent postings provide less textual content.

**salary_provided:**
- Binary flag: 1 if salary_range was provided, 0 if missing.
- Counterintuitively, postings with a salary have a *higher* fraud rate: 223 / 2,868 × 100 = 7.78% vs 643 / 15,012 × 100 = 4.28%.
- This goes against the "less info means more fraud" pattern, suggesting fraudsters may use salary as bait.

**has_company_profile:**
- Binary flag: 1 if company_profile is non-empty, 0 if empty.
- Without profile: 587 / 3,308 × 100 = 17.74% fraud rate (3.67x lift). With profile: 279 / 14,572 × 100 = 1.91%.
- 587 / 866 × 100 = 67.78% of all fraud cases lack a company profile, the strongest single feature alongside has_company_logo.

**Engineered features to carry forward:** missing_count, total_text_len, company_profile_len, description_len, requirements_len, benefits_len, company_profile_word_count, description_word_count, requirements_word_count, benefits_word_count, salary_provided, has_company_profile.

### Prepping Dataset

This section will serve to prepare the clean dataset to be used for training.

In [ ]:
# 1. Strip whitespace from the strings
df['country'] = df['country'].str.strip()
df['state'] = df['state'].str.strip()
df['city'] = df['city'].str.strip()

# 2. Replace empty strings with actual NaN objects
import numpy as np
df[['country', 'state', 'city']] = df[['country', 'state', 'city']].replace('', np.nan)

# 3. NOW fill the NaNs
df["country"] = df["country"].fillna("Not Provided")
df["state"] = df["state"].fillna("Not Provided")
df["city"] = df["city"].fillna("Not Provided")
df[["country","state","city"]].head()

In [ ]:
import pandas as pd
import re
import html
from bs4 import BeautifulSoup

def remove_terminators(text):
    if isinstance(text, str):
        return re.sub(r'\s+', ' ', text).strip()
    return text

def remove_url(text):
    if isinstance(text,str):
        return re.sub(r'#URL_[a-zA-Z0-9]+', 'link', text)
    return text

def remove_email(text):
    if isinstance(text,str):
        return re.sub(r'#EMAIL_[a-zA-Z0-9]+', 'link', text)
    return text

def remove_phone(text):
    if isinstance(text,str):
        return re.sub(r'#PHONE_[a-zA-Z0-9]+', 'link', text)
    return text

def remove_non_english_words(text):
    if not isinstance(text, str):
        return text
    # Keep only tokens that contain at least one latin/common character
    # Strips tokens that are purely Arabic, Korean, Chinese, etc.
    tokens = text.split()
    filtered = [w for w in tokens if re.search(r'[a-zA-Z0-9]', w)]
    return ' '.join(filtered)


def add_word_boundaries(text):
    if not isinstance(text, str):
        return ""
    # ALLCAPS + TitleCase → "CUSTOMER Be", "IT Service"
    text = re.sub(r'([A-Z]{2,})([A-Z][a-z])', r'\1 \2', text)
    # TitleCase + TitleCase → "N Service Bus", "GED Customer"
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    # letter followed by digit e.g. "experience6" → "experience 6"
    text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)
    # punctuation followed by letter e.g. "service.experience" → "service. experience"
    text = re.sub(r'([.,!?;:])([a-zA-Z])', r'\1 \2', text)
    return text

def split_merged_words(text):
    if not isinstance(text, str):
        return text
    # ALLCAPS + TitleCase → "CUSTOMER Be", "IT Service"
    text = re.sub(r'([A-Z]{2,})([A-Z][a-z])', r'\1 \2', text)
    # TitleCase + TitleCase → "N Service Bus", "GED Customer"
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    return text

def clean_bullet_artifacts(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'\b\d+\.\s*', ' ', text)        
    text = re.sub(r'\.\d+\.', ' ', text)           
    text = re.sub(r'[•\-–—]+\s*', ' ', text)       
    return text



def clean_job_text(text):
    if not isinstance(text, str):
        return ""


    text = html.unescape(text)
    

    text = BeautifulSoup(text, "html.parser").get_text()

    text = add_word_boundaries(text)


    text = text.lower()


    text = text.replace("not provided", " ")


    contractions = {"we're": "we are", "don't": "do not", "it's": "it is", "we've": "we have"}
    for word, replacement in contractions.items():
        text = text.replace(word, replacement)


    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', ' ', text)


    text = re.sub(r'\s+', ' ', text).strip()

    return text

def clean_job_text_pipeline(text):
    text = remove_url(text)
    text = remove_email(text)
    text = remove_phone(text)
    text = add_word_boundaries(text)
    text = clean_bullet_artifacts(text)
    text = remove_non_english_words(text)
    text = remove_terminators(text)
    text = clean_job_text(text)
    return text



text_cols = ['title', 'country', 'state', 'city', 'company_profile', 'description', 'requirements', 'benefits',"employment_type", "required_experience", "required_education", "industry", "function"]
for col in text_cols:
    df[col] = df[col].apply(clean_job_text_pipeline)


In [ ]:
df["employment_type"].value_counts()
df["required_education"].value_counts()

In [ ]:
### Prepping text columns for as free text datasets for future use in NLP

text_cols = ['title', 'country', 'state', 'city', 'company_profile', 'description', 'requirements', 'benefits',"employment_type", "required_experience", "required_education", "industry", "function"]
df["full_text"] = df[text_cols].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1)
df["full_text"] = df["full_text"].apply(clean_job_text_pipeline)


df['full_text'].head(1)

NLP_DF = df[['full_text', 'fraudulent']]
NLP_DF.head()

NLP_DF.to_csv('data/clean/fake_job_postings_nlp.csv', index=False)



In [ ]:
df.columns.tolist()

In [ ]:
df["vague_location"].value_counts()


In [ ]:
### Prepping numeric columns for future use in logistic regression if needed

numeric_cols = ["telecommuting", "missing_count", "total_text_len", "company_profile_len", "description_len", "requirements_len", "benefits_len", "company_profile_word_count",\
    "description_word_count", "requirements_word_count", "benefits_word_count", "salary_provided", "has_company_profile"\
        ,"vague_location","has_company_logo","has_questions"]

numeric_df = df[numeric_cols + ['fraudulent']]
numeric_df.shape


# numeric_df.to_csv('data/clean/fake_job_postings_numeric.csv', index=False)


In [ ]:
numeric_df.head()

In [ ]:
## Check the statistics of the numeric columns to see if there are any outliers that need to be removed before modeling

non_binary_cols = ["total_text_len", "company_profile_len", "description_len", "requirements_len", "benefits_len", "company_profile_word_count",\
    "description_word_count", "requirements_word_count", "benefits_word_count"]
for col in non_binary_cols:
    q1 = numeric_df[col].quantile(0.25)
    q3 = numeric_df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = numeric_df[(numeric_df[col] < lower_bound) | (numeric_df[col] > upper_bound)]
    ## Remove outliers from the dataset
    numeric_df = numeric_df[(numeric_df[col] >= lower_bound) & (numeric_df[col] <= upper_bound)]
    print(f"{col} has {len(outliers)} outliers")
print(numeric_df.shape)
numeric_df.head()

In [ ]:
## Check fraud split

numeric_df['fraudulent'].value_counts()

In [ ]:
numeric_df.to_csv('data/clean/fake_job_postings_numeric.csv', index=False)